# ECG + SaO₂ sleep-apnea prototype — streamed PhysioNet training

This notebook downloads a **configurable subset** of the PhysioNet/Computing in Cardiology Challenge 2018 training set directly in Colab. It never downloads the full 135 GB training directory.

For each selected subject it downloads only the header, signal matrix, and detailed WFDB annotation; extracts ECG, SaO₂, and apnea/hypopnea windows; saves a compact feature cache to Google Drive; and deletes the raw subject immediately. Completed subjects are skipped when the notebook is restarted.

The notebook then creates leakage-free subject partitions, trains the central model, evaluates untouched subjects, simulates one two-client FedAvg round for Raspberry Pi and Renesas, and exports an INT8 model and C header.

**Research prototype:** this is not a medical device or a clinical diagnosis system.


## 0. Install dependencies and choose the subset size

Start with \`SUBJECT_LIMIT = 100\`. Increasing it to 200 improves subject diversity but transfers more data. Only one raw subject (usually about 100–200 MB) is retained in Colab at a time.


In [ ]:
%pip install -q "tensorflow>=2.15" "wfdb>=4.1" "scipy>=1.10" pandas numpy matplotlib seaborn scikit-learn requests tqdm


In [ ]:
import gc
import json
import math
import random
import re
import shutil
import time
import warnings
from fractions import Fraction
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
import tensorflow as tf
import wfdb

from scipy.signal import butter, find_peaks, resample_poly, sosfiltfilt
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    confusion_matrix, f1_score, precision_recall_curve, precision_score,
    recall_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import StratifiedGroupKFold
from tqdm.auto import tqdm

SEED = 42
SUBJECT_LIMIT = 100             # Start here; use 200 after the complete run succeeds.
WINDOW_SEC = 60
STRIDE_SEC = 30
ECG_FS = 100
SPO2_FS = 1
MIN_EVENT_OVERLAP_SEC = 5.0
MIN_VALID_FRACTION = 0.90
CACHE_VERSION = "cinc2018_v1_w60_s30_feature21"

# Conservative head-only federation. The server validation set decides whether to accept an update.
LOCAL_EPOCHS = 1
LOCAL_LEARNING_RATE = 1e-4
FEDERATED_ROUNDS = 1
FEDERATION_TOLERANCE = 0.01
MIN_FEDERATED_SPREAD_RATIO = 0.50

# Prototype release gates. These label artifacts; they never select a model using final-test labels.
MAX_INT8_VALIDATION_AUC_DROP = 0.02
MIN_INT8_VALIDATION_AGREEMENT = 0.95
MAX_INT8_VALIDATION_DECISION_DROP = 0.03
MAX_INT8_VALIDATION_SENSITIVITY_DROP = 0.10
MIN_FINAL_ROC_AUC = 0.70
MIN_FINAL_BALANCED_ACCURACY = 0.60
MIN_FINAL_SENSITIVITY = 0.50
MIN_FINAL_PR_GAIN = 0.10

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
warnings.filterwarnings("ignore", category=RuntimeWarning)

PHYSIONET_ROOTS = [
    "https://physionet-open.s3.amazonaws.com/challenge-2018/1.0.0",
    "https://s3.amazonaws.com/physionet-open/challenge-2018/1.0.0",
    "https://physionet.org/files/challenge-2018/1.0.0",
]
RECORDS_URLS = [f"{root}/training/RECORDS" for root in PHYSIONET_ROOTS]
RAW_ROOT = Path("/content/cinc2018_current_subject")
ARTIFACT_DIR = Path("/content/cinc2018_apnea_artifacts")

print("TensorFlow:", tf.__version__)
print("WFDB:", wfdb.__version__)
print("Requested subjects:", SUBJECT_LIMIT)


## 1. Mount Drive and create the resumable cache

Change \`DRIVE_PROJECT_DIR\` only if you want a different Drive folder. No dataset ZIP is required.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/apena_ml_challenge2018")
CACHE_DIR = DRIVE_PROJECT_DIR / "processed_features"
MANIFEST_PATH = DRIVE_PROJECT_DIR / "processing_manifest.csv"
SPLIT_PATH = DRIVE_PROJECT_DIR / "subject_split_manifest.csv"

# Every write goes to local Colab storage first. Drive receives an atomic copy when its mount is healthy.
LOCAL_STATE_DIR = Path("/content/apena_ml_local_state")
LOCAL_CACHE_DIR = LOCAL_STATE_DIR / "processed_features"
LOCAL_MANIFEST_PATH = LOCAL_STATE_DIR / "processing_manifest.csv"
LOCAL_SPLIT_PATH = LOCAL_STATE_DIR / "subject_split_manifest.csv"
for directory in (LOCAL_STATE_DIR, LOCAL_CACHE_DIR, RAW_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

def drive_available():
    try:
        my_drive = Path("/content/drive/MyDrive")
        if not my_drive.is_dir():
            return False
        DRIVE_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
        CACHE_DIR.mkdir(parents=True, exist_ok=True)
        return True
    except OSError:
        return False

def path_exists_safely(path):
    try:
        return Path(path).exists()
    except OSError:
        return False

def copy_file_to_drive(source, destination):
    source, destination = Path(source), Path(destination)
    if not drive_available():
        return False
    temporary = destination.with_suffix(destination.suffix + ".part")
    try:
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, temporary)
        temporary.replace(destination)
        return True
    except OSError as error:
        print(f"Drive copy postponed for {source.name}: {error}")
        return False

def write_csv_resilient(frame, local_path, drive_path, **kwargs):
    local_path = Path(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = local_path.with_suffix(local_path.suffix + ".part")
    frame.to_csv(temporary, **kwargs)
    temporary.replace(local_path)
    return copy_file_to_drive(local_path, drive_path)

if drive_available():
    print("Persistent Drive cache:", CACHE_DIR)
else:
    print("WARNING: Drive is currently unavailable; local staging will be used.")
print("Local staging cache:", LOCAL_CACHE_DIR)
print("Raw temporary area:", RAW_ROOT)
print("Free Colab disk: %.1f GiB" % (shutil.disk_usage("/content").free / 1024**3))


## 2. Select subjects from the official training manifest

Selection is deterministic. The same seed and limit always choose the same subjects, so interrupted runs resume against the same cache.


In [ ]:
HTTP_HEADERS = {"User-Agent": "apena-ml-colab/1.0 (PhysioNet research prototype)"}

def http_text(urls, attempts=4, timeout=60):
    errors = []
    for attempt in range(1, attempts + 1):
        for url in urls:
            try:
                response = requests.get(url, headers=HTTP_HEADERS, timeout=timeout)
                response.raise_for_status()
                if response.text.strip():
                    print("Manifest source:", url)
                    return response.text
                raise RuntimeError("empty response")
            except (requests.RequestException, RuntimeError) as error:
                errors.append(f"{url}: {type(error).__name__}: {error}")
        if attempt < attempts:
            wait_seconds = min(30, 2 ** attempt)
            print(f"All manifest mirrors failed (attempt {attempt}/{attempts}); retrying in {wait_seconds}s")
            time.sleep(wait_seconds)
    raise RuntimeError("Could not download the PhysioNet manifest from any official mirror:\n" + "\n".join(errors[-6:]))

manifest_text = http_text(RECORDS_URLS)
record_lines = [line.strip().rstrip("/") for line in manifest_text.splitlines() if line.strip()]
all_subjects = sorted({Path(line).name for line in record_lines})
if SUBJECT_LIMIT > len(all_subjects):
    raise ValueError(f"SUBJECT_LIMIT={SUBJECT_LIMIT} exceeds the {len(all_subjects)} labelled training subjects")

rng = np.random.default_rng(SEED)
SUBJECT_IDS = sorted(rng.permutation(all_subjects)[:SUBJECT_LIMIT].tolist())
print(f"Official labelled subjects: {len(all_subjects)}")
print(f"Selected: {len(SUBJECT_IDS)}")
print("First ten:", SUBJECT_IDS[:10])


## 3. Download and label one subject at a time

The detailed \`.arousal\` annotation contains paired entries such as \`resp_obstructiveapnea\`, \`resp_centralapnea\`, and \`resp_hypopnea\`. The \`-arousal.mat\` challenge target is intentionally **not used**, because it represents non-apnea arousals rather than apnea labels.


In [ ]:
FEATURE_NAMES = [
    "rr_count", "mean_hr", "std_hr", "mean_rr", "sdnn", "rmssd", "pnn50", "rr_cv",
    "ecg_mean_abs", "ecg_std", "ecg_range",
    "spo2_mean", "spo2_min", "spo2_std", "spo2_range", "spo2_end_minus_start",
    "spo2_max_drop", "spo2_diff_std", "spo2_below_90", "spo2_below_92", "spo2_below_94",
]
ECG_FEATURE_INDEX = np.arange(0, 11)
SPO2_FEATURE_INDEX = np.arange(11, len(FEATURE_NAMES))

def subject_urls(subject, suffix):
    return [f"{root}/training/{subject}/{subject}{suffix}" for root in PHYSIONET_ROOTS]

def download_atomic(urls, destination, attempts=4):
    destination = Path(destination)
    temporary = destination.with_suffix(destination.suffix + ".part")
    errors = []
    for attempt in range(1, attempts + 1):
        for url in urls:
            try:
                existing = temporary.stat().st_size if temporary.exists() else 0
                headers = dict(HTTP_HEADERS)
                if existing:
                    headers["Range"] = f"bytes={existing}-"
                with requests.get(url, headers=headers, stream=True, timeout=(30, 180)) as response:
                    if existing and response.status_code == 200:
                        temporary.unlink(missing_ok=True)
                        existing = 0
                    response.raise_for_status()
                    mode = "ab" if existing and response.status_code == 206 else "wb"
                    total = response.headers.get("Content-Length")
                    total = int(total) + existing if total is not None else None
                    with open(temporary, mode) as handle, tqdm(
                        total=total, initial=existing, unit="B", unit_scale=True,
                        desc=destination.name, leave=False,
                    ) as progress:
                        for chunk in response.iter_content(chunk_size=4 * 1024 * 1024):
                            if chunk:
                                handle.write(chunk)
                                progress.update(len(chunk))
                temporary.replace(destination)
                return destination
            except (requests.RequestException, OSError) as error:
                errors.append(f"{url}: {type(error).__name__}: {error}")
        if attempt < attempts:
            wait_seconds = min(30, 2 ** attempt)
            print(f"All file mirrors failed for {destination.name} (attempt {attempt}/{attempts}); retrying in {wait_seconds}s")
            time.sleep(wait_seconds)
    raise RuntimeError(
        f"Could not download {destination.name} from any official mirror:\n" + "\n".join(errors[-6:])
    )
def drive_cache_path(subject):
    return CACHE_DIR / f"{subject}_{CACHE_VERSION}.npz"

def local_cache_path(subject):
    return LOCAL_CACHE_DIR / f"{subject}_{CACHE_VERSION}.npz"

def cache_path(subject):
    persistent = drive_cache_path(subject)
    if path_exists_safely(persistent):
        return persistent
    return local_cache_path(subject)

def interpolate_invalid(values, valid):
    values = np.asarray(values, dtype=np.float64)
    valid = np.asarray(valid, dtype=bool)
    if valid.sum() < 2:
        raise ValueError("fewer than two valid samples")
    index = np.arange(len(values))
    return np.interp(index, index[valid], values[valid])

def resample_to(values, source_fs, target_fs):
    ratio = Fraction(float(target_fs) / float(source_fs)).limit_denominator(2000)
    return resample_poly(values, ratio.numerator, ratio.denominator)

def respiratory_events(record_base, sample_rate):
    annotation = wfdb.rdann(str(record_base), "arousal")
    stacks, events = {}, []
    for sample, note in zip(annotation.sample, annotation.aux_note):
        note = str(note).strip().lower()
        if note.startswith("("):
            event_type = note[1:]
            if event_type.startswith("resp_") and ("apnea" in event_type or "hypopnea" in event_type):
                stacks.setdefault(event_type, []).append(float(sample) / sample_rate)
        elif note.endswith(")"):
            event_type = note[:-1]
            if event_type in stacks and stacks[event_type]:
                start = stacks[event_type].pop()
                end = float(sample) / sample_rate
                if end > start:
                    events.append({"start": start, "end": end, "type": event_type})
    return sorted(events, key=lambda event: event["start"])

def maximum_event_overlap(start, end, events):
    overlap = 0.0
    for event in events:
        if event["start"] >= end:
            break
        if event["end"] > start:
            overlap = max(overlap, min(end, event["end"]) - max(start, event["start"]))
    return overlap

def extract_window_features(ecg, spo2):
    peak_options = []
    for candidate in (ecg, -ecg):
        peaks, _ = find_peaks(candidate, distance=int(0.30 * ECG_FS), prominence=0.45)
        rr = np.diff(peaks) / ECG_FS
        rr = rr[(rr >= 0.30) & (rr <= 2.0)]
        peak_options.append(rr)
    rr = max(peak_options, key=len)
    if len(rr) >= 3:
        heart_rate = 60.0 / rr
        rr_diff = np.diff(rr)
        rr_features = [
            len(rr), np.mean(heart_rate), np.std(heart_rate), np.mean(rr), np.std(rr),
            np.sqrt(np.mean(rr_diff ** 2)) if len(rr_diff) else 0.0,
            np.mean(np.abs(rr_diff) > 0.05) if len(rr_diff) else 0.0,
            np.std(rr) / (np.mean(rr) + 1e-6),
        ]
    else:
        rr_features = [len(rr), 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

    running_high = np.maximum.accumulate(spo2)
    ecg_features = [np.mean(np.abs(ecg)), np.std(ecg), np.ptp(ecg)]
    spo2_features = [
        np.mean(spo2), np.min(spo2), np.std(spo2), np.ptp(spo2),
        np.mean(spo2[-5:]) - np.mean(spo2[:5]), np.max(running_high - spo2),
        np.std(np.diff(spo2)) if len(spo2) > 1 else 0.0,
        np.mean(spo2 < 90), np.mean(spo2 < 92), np.mean(spo2 < 94),
    ]
    result = np.asarray(rr_features + ecg_features + spo2_features, dtype=np.float32)
    return np.nan_to_num(result, nan=0.0, posinf=0.0, neginf=0.0)


In [ ]:
def process_subject(subject, force=False):
    output = cache_path(subject)
    if path_exists_safely(output) and not force:
        if output == local_cache_path(subject):
            persistent = drive_cache_path(subject)
            if copy_file_to_drive(output, persistent):
                output = persistent
        return output, "cached"

    subject_dir = RAW_ROOT / subject
    if subject_dir.exists():
        shutil.rmtree(subject_dir)
    subject_dir.mkdir(parents=True)
    record_base = subject_dir / subject

    try:
        for suffix in (".hea", ".arousal", ".mat"):
            download_atomic(subject_urls(subject, suffix), record_base.with_suffix(suffix))

        header = wfdb.rdheader(str(record_base))
        signal_names = list(header.sig_name)
        name_map = {name.strip().lower(): name for name in signal_names}
        if "ecg" not in name_map or "sao2" not in name_map:
            raise ValueError(f"required channels absent; available={signal_names}")

        record = wfdb.rdrecord(str(record_base), channel_names=[name_map["ecg"], name_map["sao2"]])
        native_fs = float(record.fs)
        ecg_native = np.asarray(record.p_signal[:, 0], dtype=np.float64)
        spo2_native = np.asarray(record.p_signal[:, 1], dtype=np.float64)
        del record

        ecg_valid_native = np.isfinite(ecg_native)
        spo2_valid_native = np.isfinite(spo2_native) & (spo2_native >= 50.0) & (spo2_native <= 100.0)
        ecg = resample_to(interpolate_invalid(ecg_native, ecg_valid_native), native_fs, ECG_FS)
        spo2 = resample_to(interpolate_invalid(spo2_native, spo2_valid_native), native_fs, SPO2_FS)
        ecg_valid = resample_to(ecg_valid_native.astype(float), native_fs, ECG_FS) >= 0.90
        spo2_valid = resample_to(spo2_valid_native.astype(float), native_fs, SPO2_FS) >= 0.90
        del ecg_native, spo2_native

        ecg_filter = butter(4, [0.5, 35.0], btype="bandpass", fs=ECG_FS, output="sos")
        ecg = sosfiltfilt(ecg_filter, ecg).astype(np.float32)
        spo2 = np.clip(spo2, 50.0, 100.0).astype(np.float32)
        events = respiratory_events(record_base, native_fs)

        duration_sec = min(len(ecg) / ECG_FS, len(spo2) / SPO2_FS)
        n_windows = max(0, int((duration_sec - WINDOW_SEC) // STRIDE_SEC) + 1)
        features, labels, starts = [], [], []
        dropped_quality = dropped_ambiguous = 0

        for window_index in range(n_windows):
            start = float(window_index * STRIDE_SEC)
            end = start + WINDOW_SEC
            ecg_slice = slice(int(start * ECG_FS), int(start * ECG_FS) + WINDOW_SEC * ECG_FS)
            spo2_slice = slice(int(start * SPO2_FS), int(start * SPO2_FS) + WINDOW_SEC * SPO2_FS)
            ecg_window, spo2_window = ecg[ecg_slice], spo2[spo2_slice]
            if len(ecg_window) != WINDOW_SEC * ECG_FS or len(spo2_window) != WINDOW_SEC * SPO2_FS:
                continue
            if ecg_valid[ecg_slice].mean() < MIN_VALID_FRACTION or spo2_valid[spo2_slice].mean() < MIN_VALID_FRACTION:
                dropped_quality += 1
                continue
            ecg_scale = float(np.std(ecg_window))
            if not np.isfinite(ecg_scale) or ecg_scale < 1e-10:
                dropped_quality += 1
                continue
            overlap = maximum_event_overlap(start, end, events)
            if 0.0 < overlap < MIN_EVENT_OVERLAP_SEC:
                dropped_ambiguous += 1
                continue

            ecg_normalized = np.clip((ecg_window - np.median(ecg_window)) / (ecg_scale + 1e-8), -5, 5)
            features.append(extract_window_features(ecg_normalized, spo2_window))
            labels.append(int(overlap >= MIN_EVENT_OVERLAP_SEC))
            starts.append(start)

        if not features or len(np.unique(labels)) < 2:
            raise RuntimeError("subject produced no usable two-class windows")

        temporary_output = output.with_suffix(".npz.part")
        with open(temporary_output, "wb") as handle:
            np.savez_compressed(
                handle, features=np.stack(features).astype(np.float32),
                y=np.asarray(labels, dtype=np.int8), start_sec=np.asarray(starts, dtype=np.float32),
                subject=np.asarray(subject), event_count=np.asarray(len(events), dtype=np.int32),
                native_fs=np.asarray(native_fs, dtype=np.float32),
                dropped_quality=np.asarray(dropped_quality, dtype=np.int32),
                dropped_ambiguous=np.asarray(dropped_ambiguous, dtype=np.int32),
                feature_names=np.asarray(FEATURE_NAMES),
            )
        temporary_output.replace(output)
        persistent = drive_cache_path(subject)
        if output != persistent and copy_file_to_drive(output, persistent):
            output = persistent
        return output, "processed"
    finally:
        shutil.rmtree(subject_dir, ignore_errors=True)
        gc.collect()


## 4. Build or resume the compact Drive cache

This is the only slow stage. A Colab disconnect is safe: rerun from the top and completed \`.npz\` subjects are skipped. Failures are recorded and retried on the next run.


In [ ]:
manifest_rows = []
drive_was_available = drive_available()
for index, subject in enumerate(SUBJECT_IDS, start=1):
    print(f"[{index:03d}/{len(SUBJECT_IDS)}] {subject}")
    row = {"subject": subject, "status": "failed", "message": "", "cache_location": "none"}
    try:
        path, state = process_subject(subject)
        with np.load(path, allow_pickle=False) as data:
            y = data["y"]
            row.update({
                "status": state, "windows": len(y), "positive_windows": int(y.sum()),
                "positive_rate": float(y.mean()), "event_count": int(data["event_count"]),
                "cache_mib": path.stat().st_size / 1024**2,
                "cache_location": "drive" if path == drive_cache_path(subject) else "local",
            })
    except Exception as error:
        row["message"] = f"{type(error).__name__}: {error}"
        print("  FAILED:", row["message"])
    manifest_rows.append(row)
    manifest_frame = pd.DataFrame(manifest_rows)
    manifest_synced = write_csv_resilient(manifest_frame, LOCAL_MANIFEST_PATH, MANIFEST_PATH, index=False)
    if drive_was_available and not manifest_synced:
        print("WARNING: Google Drive disconnected. Processing continues in local staging.")
        print("Remount Drive before the runtime ends, then rerun this cell to synchronize local caches.")
        drive_was_available = False

manifest = pd.DataFrame(manifest_rows)
display(manifest)
successful_subjects = manifest.loc[manifest.status.isin(["cached", "processed"]), "subject"].tolist()
local_only_subjects = [subject for subject in successful_subjects if not path_exists_safely(drive_cache_path(subject))]
print(f"Successful: {len(successful_subjects)}/{len(SUBJECT_IDS)}")
print("Available feature cache: %.1f MiB" % (sum(cache_path(s).stat().st_size for s in successful_subjects) / 1024**2))
print("Unsynced local subjects:", len(local_only_subjects))
if local_only_subjects:
    print("Keep this runtime active. Remount Drive and rerun this cell to copy these caches:")
    print(local_only_subjects)
if len(successful_subjects) < 25:
    raise RuntimeError("Fewer than 25 subjects were processed; inspect the failure messages before training")


## 5. Create fixed subject-level partitions

No patient can appear in more than one partition. With 100 successful subjects the target split is 70 central training, 10 server validation, 10 untouched final test, 5 Pi clients, and 5 Renesas clients. The search balances window prevalence, respiratory-event density, and recording cohort (\`tr03\`, \`tr04\`, etc.) to reduce distribution shift.


In [ ]:
subject_stats = manifest.loc[manifest.status.isin(["cached", "processed"])].copy()
subject_stats["negative_windows"] = subject_stats.windows - subject_stats.positive_windows
subject_stats["record_group"] = subject_stats.subject.str.extract(r"^(tr\d+)", expand=False)
subject_stats["events_per_1000_windows"] = 1000.0 * subject_stats.event_count / subject_stats.windows
subject_stats = subject_stats.sort_values("subject").reset_index(drop=True)

def proportional_partition_sizes(total):
    ratios = {"central_train": 0.70, "validation": 0.10, "final_test": 0.10, "pi_client": 0.05}
    sizes = {key: max(1, int(round(total * value))) for key, value in ratios.items()}
    sizes["renesas_client"] = total - sum(sizes.values())
    while sizes["renesas_client"] < 1:
        largest = max((key for key in sizes if key != "renesas_client"), key=sizes.get)
        sizes[largest] -= 1
        sizes["renesas_client"] += 1
    return sizes

PARTITION_SIZES = proportional_partition_sizes(len(subject_stats))
stats_by_subject = subject_stats.set_index("subject")
global_rate = subject_stats.positive_windows.sum() / subject_stats.windows.sum()
global_event_density = subject_stats.event_count.sum() / subject_stats.windows.sum()
global_group_distribution = subject_stats.record_group.value_counts(normalize=True)
rng = np.random.default_rng(SEED)
best_assignment, best_score = None, float("inf")

for _ in range(30000):
    order = list(rng.permutation(subject_stats.subject.to_numpy()))
    assignment, cursor, score, valid = {}, 0, 0.0, True
    for partition, count in PARTITION_SIZES.items():
        members = order[cursor:cursor + count]
        cursor += count
        table = stats_by_subject.loc[members]
        positives, windows = table.positive_windows.sum(), table.windows.sum()
        if positives == 0 or positives == windows:
            valid = False
            break
        positive_rate = positives / windows
        event_density = table.event_count.sum() / windows
        group_distribution = table.record_group.value_counts(normalize=True)
        group_distance = sum(
            abs(group_distribution.get(group, 0.0) - global_group_distribution.get(group, 0.0))
            for group in global_group_distribution.index
        )
        score += abs(positive_rate - global_rate)
        score += 0.25 * abs(event_density - global_event_density) / max(global_event_density, 1e-8)
        score += 0.20 * group_distance
        assignment[partition] = members
    if valid and score < best_score:
        best_assignment, best_score = assignment, score

if best_assignment is None:
    raise RuntimeError("Could not construct balanced two-class subject partitions")

partition_lookup = {subject: part for part, members in best_assignment.items() for subject in members}
subject_stats["partition"] = subject_stats.subject.map(partition_lookup)
split_synced = write_csv_resilient(subject_stats, LOCAL_SPLIT_PATH, SPLIT_PATH, index=False)
if not split_synced:
    print("WARNING: split manifest is local only because Drive is unavailable:", LOCAL_SPLIT_PATH)
partition_summary = subject_stats.groupby("partition").agg(
    subjects=("subject", "size"), windows=("windows", "sum"),
    positive_windows=("positive_windows", "sum"), events=("event_count", "sum"),
    recording_groups=("record_group", lambda values: ",".join(sorted(set(values)))),
)
partition_summary["positive_rate"] = partition_summary.positive_windows / partition_summary.windows
partition_summary["events_per_1000_windows"] = 1000.0 * partition_summary.events / partition_summary.windows
display(partition_summary)
display(subject_stats[["subject", "record_group", "partition", "windows", "positive_rate", "events_per_1000_windows"]])


In [ ]:
def load_partition(partition):
    feature_parts, label_parts, subject_parts, start_parts = [], [], [], []
    members = subject_stats.loc[subject_stats.partition == partition, "subject"]
    for subject in members:
        with np.load(cache_path(subject), allow_pickle=False) as data:
            count = len(data["y"])
            feature_parts.append(data["features"])
            label_parts.append(data["y"].astype(np.float32))
            subject_parts.append(np.full(count, subject))
            start_parts.append(data["start_sec"])
    return tuple(map(np.concatenate, (feature_parts, label_parts, subject_parts, start_parts)))

F_train, y_train, subject_train, start_train = load_partition("central_train")
F_val, y_val, subject_val, start_val = load_partition("validation")
F_test, y_test, subject_test, start_test = load_partition("final_test")
F_pi, y_pi, subject_pi, start_pi = load_partition("pi_client")
F_ra, y_ra, subject_ra, start_ra = load_partition("renesas_client")

for name, features, labels, subjects in [
    ("central_train", F_train, y_train, subject_train), ("validation", F_val, y_val, subject_val),
    ("final_test", F_test, y_test, subject_test), ("pi_client", F_pi, y_pi, subject_pi),
    ("renesas_client", F_ra, y_ra, subject_ra),
]:
    print(name, features.shape, "subjects=", len(np.unique(subjects)), "positive_rate=", round(float(labels.mean()), 3))

if not np.isfinite(np.concatenate([F_train, F_val, F_test, F_pi, F_ra])).all():
    raise AssertionError("Non-finite feature value found")


## 6. ECG-only, SaO₂-only, and combined ablation

The deployment model is always the combined 21-feature model. The single-sensor models are evidence that both sensors contribute; they are not deployment candidates.


In [ ]:
def feature_view(name, features):
    if name == "ecg_only":
        return features[:, ECG_FEATURE_INDEX]
    if name == "spo2_only":
        return features[:, SPO2_FEATURE_INDEX]
    if name == "feature_mlp":
        return features
    raise ValueError(name)

def patient_sample_weights(y, subjects):
    weights = np.zeros(len(y), dtype=np.float32)
    unique_subjects = np.unique(subjects)
    for subject in unique_subjects:
        mask = subjects == subject
        labels = y[mask].astype(int)
        counts = np.bincount(labels, minlength=2)
        for label in (0, 1):
            label_mask = mask & (y == label)
            if counts[label]:
                weights[label_mask] = 1.0 / (len(unique_subjects) * 2.0 * counts[label])
    return weights / weights.mean()

def build_model(name, reference):
    tf.keras.backend.clear_session()
    normalizer = tf.keras.layers.Normalization(axis=-1, name="feature_normalization")
    normalizer.adapt(reference)
    inputs = tf.keras.Input(shape=(reference.shape[1],), name="features")
    x = normalizer(inputs)
    x = tf.keras.layers.Dense(32, activation="relu", name="shared_dense_1")(x)
    x = tf.keras.layers.Dropout(0.15)(x)
    x = tf.keras.layers.Dense(16, activation="relu", name="shared_dense_2")(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid", name="federated_head")(x)
    return tf.keras.Model(inputs, outputs, name=name)

def compile_model(model, learning_rate=1e-3):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate), loss="binary_crossentropy",
        metrics=[tf.keras.metrics.AUC(curve="ROC", name="roc_auc"), tf.keras.metrics.AUC(curve="PR", name="pr_auc")],
    )

def predict_probability(model, features):
    return model.predict(features, batch_size=1024, verbose=0).reshape(-1)

def ranking_metrics_by_patient(y, probability, subjects):
    rows = []
    for subject in np.unique(subjects):
        mask = subjects == subject
        if len(np.unique(y[mask])) < 2:
            continue
        rows.append({
            "subject": subject, "windows": int(mask.sum()), "positive_rate": float(y[mask].mean()),
            "roc_auc": roc_auc_score(y[mask], probability[mask]),
            "pr_auc": average_precision_score(y[mask], probability[mask]),
        })
    return pd.DataFrame(rows)

def fit_with_validation(name, train_features, train_y, train_subjects, val_features, val_y, tag):
    tf.keras.utils.set_random_seed(SEED + sum(ord(char) for char in tag))
    model = build_model(name, train_features)
    compile_model(model)
    checkpoint = f"/content/{tag}_{name}.keras"
    history = model.fit(
        train_features, train_y, sample_weight=patient_sample_weights(train_y, train_subjects),
        validation_data=(val_features, val_y), epochs=40, batch_size=256, verbose=0,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(monitor="val_pr_auc", mode="max", patience=6, restore_best_weights=True),
            tf.keras.callbacks.ModelCheckpoint(checkpoint, monitor="val_pr_auc", mode="max", save_best_only=True),
        ],
    )
    best_epoch = int(np.argmax(history.history["val_pr_auc"]) + 1)
    return model, history, best_epoch, checkpoint


In [ ]:
candidate_rows = []
ablation_checkpoints = {}
for name in ("ecg_only", "spo2_only", "feature_mlp"):
    train_view, val_view = feature_view(name, F_train), feature_view(name, F_val)
    model, history, best_epoch, checkpoint = fit_with_validation(
        name, train_view, y_train, subject_train, val_view, y_val, f"ablation_{name}",
    )
    probability = predict_probability(model, val_view)
    patient_table = ranking_metrics_by_patient(y_val, probability, subject_val)
    candidate_rows.append({
        "model": name, "best_epoch": best_epoch,
        "pooled_roc_auc": roc_auc_score(y_val, probability),
        "pooled_pr_auc": average_precision_score(y_val, probability),
        "mean_patient_roc_auc": patient_table.roc_auc.mean(),
        "mean_patient_pr_auc": patient_table.pr_auc.mean(),
    })
    ablation_checkpoints[name] = checkpoint
    del model
    gc.collect()

candidate_results = pd.DataFrame(candidate_rows).set_index("model")
display(candidate_results.round(4))
selected_name = "feature_mlp"
print("Deployment architecture:", selected_name, "(uses ECG + SaO2)")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
candidate_results["mean_patient_roc_auc"].plot.bar(ax=axes[0], color="tab:blue")
axes[0].axhline(0.5, color="gray", linestyle="--", label="random ROC=0.5")
axes[0].set(title="Mean validation patient ROC-AUC", ylabel="ROC-AUC")
axes[0].legend()
candidate_results["mean_patient_pr_auc"].plot.bar(ax=axes[1], color="tab:orange")
axes[1].axhline(y_val.mean(), color="gray", linestyle="--", label=f"pooled PR baseline={y_val.mean():.3f}")
axes[1].set(title="Mean validation patient PR-AUC", ylabel="PR-AUC")
axes[1].legend()
for axis in axes:
    axis.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()


## 7. Five-fold subject cross-validation and honest diagnostics

Every development prediction is made by a fold model that did not train on that patient. The out-of-fold threshold and fold ensemble are diagnostics. They do not select the final deployable model from final-test performance.


In [ ]:
F_dev = np.concatenate([F_train, F_val])
y_dev = np.concatenate([y_train, y_val])
subject_dev = np.concatenate([subject_train, subject_val])
start_dev = np.concatenate([start_train, start_val])

group_kfold = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_probability = np.full(len(y_dev), np.nan, dtype=np.float32)
fold_records, fold_patient_rows, fold_best_epochs = [], [], []

for fold, (train_index, val_index) in enumerate(
    group_kfold.split(F_dev, y_dev, groups=subject_dev), start=1,
):
    print(f"Fold {fold}/5")
    model, _, best_epoch, checkpoint = fit_with_validation(
        selected_name, F_dev[train_index], y_dev[train_index], subject_dev[train_index],
        F_dev[val_index], y_dev[val_index], f"fold_{fold}",
    )
    fold_probability = predict_probability(model, F_dev[val_index])
    oof_probability[val_index] = fold_probability
    table = ranking_metrics_by_patient(y_dev[val_index], fold_probability, subject_dev[val_index])
    table["fold"] = fold
    fold_patient_rows.append(table)
    fold_best_epochs.append(best_epoch)
    fold_records.append({
        "fold": fold, "path": checkpoint, "best_epoch": best_epoch,
        "heldout_subjects": ",".join(sorted(np.unique(subject_dev[val_index]))),
        "mean_patient_roc_auc": float(table.roc_auc.mean()),
        "mean_patient_pr_auc": float(table.pr_auc.mean()),
    })
    del model
    gc.collect()

if np.isnan(oof_probability).any():
    raise AssertionError("Missing out-of-fold predictions")
cross_validation_patients = pd.concat(fold_patient_rows, ignore_index=True)
fold_summary = pd.DataFrame(fold_records)

oof_thresholds = np.linspace(0.05, 0.95, 181)
oof_threshold_scores = [
    balanced_accuracy_score(y_dev, oof_probability >= threshold)
    for threshold in oof_thresholds
]
oof_threshold = float(oof_thresholds[int(np.argmax(oof_threshold_scores))])

print("OOF ROC-AUC:", round(roc_auc_score(y_dev, oof_probability), 4))
print("OOF PR-AUC:", round(average_precision_score(y_dev, oof_probability), 4), "baseline:", round(float(y_dev.mean()), 4))
print("Mean patient ROC-AUC:", round(cross_validation_patients.roc_auc.mean(), 4))
print("Mean patient PR-AUC:", round(cross_validation_patients.pr_auc.mean(), 4))
print("OOF diagnostic threshold:", round(oof_threshold, 3))
display(fold_summary.drop(columns="path").round(4))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fpr, tpr, _ = roc_curve(y_dev, oof_probability)
axes[0].plot(fpr, tpr, label=f"AUC={roc_auc_score(y_dev, oof_probability):.3f}")
axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set(title="Out-of-fold ROC", xlabel="False-positive rate", ylabel="True-positive rate")
precision, recall, _ = precision_recall_curve(y_dev, oof_probability)
axes[1].plot(recall, precision, label=f"AP={average_precision_score(y_dev, oof_probability):.3f}")
axes[1].axhline(y_dev.mean(), color="gray", linestyle="--", label=f"baseline={y_dev.mean():.3f}")
axes[1].set(title="Out-of-fold precision-recall", xlabel="Recall", ylabel="Precision")
for axis in axes:
    axis.legend()
plt.tight_layout()
plt.show()


## 8. Train a validation-controlled central model

The deployable central model trains only on \`central_train\`. The server validation patients control early stopping and threshold calibration. The untouched final-test, Pi, and Renesas partitions cannot select this model.


In [ ]:
def calculate_metrics(y_true, probability, threshold):
    prediction = (probability >= threshold).astype(np.int8)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {
        "windows": int(len(y_true)), "positive_rate": float(np.mean(y_true)),
        "predicted_positive_rate": float(np.mean(prediction)),
        "accuracy": accuracy_score(y_true, prediction),
        "balanced_accuracy": balanced_accuracy_score(y_true, prediction),
        "precision": precision_score(y_true, prediction, zero_division=0),
        "sensitivity": recall_score(y_true, prediction, zero_division=0),
        "specificity": tn / (tn + fp) if tn + fp else float("nan"),
        "f1": f1_score(y_true, prediction, zero_division=0),
        "roc_auc": roc_auc_score(y_true, probability) if len(np.unique(y_true)) == 2 else float("nan"),
        "pr_auc": average_precision_score(y_true, probability),
    }

def select_balanced_threshold(y_true, probability):
    thresholds = np.linspace(0.02, 0.98, 193)
    rows = []
    for threshold in thresholds:
        prediction = probability >= threshold
        rows.append((balanced_accuracy_score(y_true, prediction), recall_score(y_true, prediction, zero_division=0), threshold))
    # Balanced accuracy is primary; sensitivity breaks numerical ties.
    return float(max(rows, key=lambda row: (row[0], row[1]))[2])

def mean_patient_ranking(y_true, probability, subjects):
    table = ranking_metrics_by_patient(y_true, probability, subjects)
    return table, float(table.roc_auc.mean()), float(table.pr_auc.mean())

# Train from scratch with a real validation checkpoint instead of a fixed median epoch count.
central_model, central_history, central_best_epoch, central_checkpoint = fit_with_validation(
    selected_name, F_train, y_train, subject_train, F_val, y_val, "central_deployment",
)
central_model = tf.keras.models.load_model(central_checkpoint)
central_validation_probability = predict_probability(central_model, F_val)
central_threshold = select_balanced_threshold(y_val, central_validation_probability)
central_validation_patients, central_val_mean_roc, central_val_mean_pr = mean_patient_ranking(
    y_val, central_validation_probability, subject_val,
)

evaluation_sets = {
    "validation": (F_val, y_val, subject_val),
    "final_test": (F_test, y_test, subject_test),
    "pi_before_federation": (F_pi, y_pi, subject_pi),
    "renesas_before_federation": (F_ra, y_ra, subject_ra),
}
central_probabilities, central_rows, central_patient_rows = {}, [], []
for partition, (features, labels, subjects) in evaluation_sets.items():
    probability = predict_probability(central_model, features)
    central_probabilities[partition] = probability
    central_rows.append({"partition": partition, **calculate_metrics(labels, probability, central_threshold)})
    table = ranking_metrics_by_patient(labels, probability, subjects)
    table["partition"] = partition
    central_patient_rows.append(table)

central_evaluation_metrics = pd.DataFrame(central_rows).set_index("partition")
central_patient_metrics = pd.concat(central_patient_rows, ignore_index=True)
print("Central best epoch:", central_best_epoch)
print("Central validation threshold:", round(central_threshold, 3))
display(central_evaluation_metrics.round(4))
display(central_patient_metrics.sort_values(["partition", "subject"]).round(4))

# Diagnostic only: average the five fold models on partitions untouched by every fold.
ensemble_sets = {
    "final_test": (F_test, y_test, subject_test),
    "pi_client": (F_pi, y_pi, subject_pi),
    "renesas_client": (F_ra, y_ra, subject_ra),
}
fold_ensemble_probabilities = {name: np.zeros(len(values[1]), dtype=np.float64) for name, values in ensemble_sets.items()}
for record in fold_records:
    fold_model = tf.keras.models.load_model(record["path"])
    for partition, (features, labels, subjects) in ensemble_sets.items():
        fold_ensemble_probabilities[partition] += predict_probability(fold_model, features) / len(fold_records)
    del fold_model
    gc.collect()

ensemble_rows = []
for partition, probability in fold_ensemble_probabilities.items():
    features, labels, subjects = ensemble_sets[partition]
    ensemble_rows.append({"partition": partition, **calculate_metrics(labels, probability, oof_threshold)})
fold_ensemble_diagnostics = pd.DataFrame(ensemble_rows).set_index("partition")
print("Five-fold ensemble diagnostic (never used for model selection)")
display(fold_ensemble_diagnostics.round(4))


In [ ]:
central_test_probability = central_probabilities["final_test"]
central_test_prediction = (central_test_probability >= central_threshold).astype(np.int8)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
sns.heatmap(confusion_matrix(y_test, central_test_prediction), annot=True, fmt="d", cmap="Blues",
            xticklabels=["Normal", "Apnea"], yticklabels=["Normal", "Apnea"], ax=axes[0])
axes[0].set(title="Central model: untouched final test", xlabel="Predicted", ylabel="Actual")
fpr, tpr, _ = roc_curve(y_test, central_test_probability)
axes[1].plot(fpr, tpr, label=f"AUC={roc_auc_score(y_test, central_test_probability):.3f}")
axes[1].plot([0, 1], [0, 1], "--", color="gray")
axes[1].set(title="Central final-test ROC", xlabel="False-positive rate", ylabel="True-positive rate")
precision, recall, _ = precision_recall_curve(y_test, central_test_probability)
axes[2].plot(recall, precision, label=f"AP={average_precision_score(y_test, central_test_probability):.3f}")
axes[2].axhline(y_test.mean(), color="gray", linestyle="--", label=f"baseline={y_test.mean():.3f}")
axes[2].set(title="Central final-test precision-recall", xlabel="Recall", ylabel="Precision")
for axis in axes[1:]:
    axis.legend()
plt.tight_layout()
plt.show()


## 9. Simulate conservative two-client FedAvg with server acceptance

Only \`federated_head\` is trainable locally. Pi and Renesas start from identical accepted server weights and never share raw patient data. A proposed FedAvg update is accepted only when mean patient ROC-AUC and PR-AUC on the server validation partition remain within the configured tolerance. Final-test labels cannot accept or reject federation.


In [ ]:
def copy_model_exact(model, copy_name):
    # Keras clone_model + set_weights changed this model's Normalization behavior.
    # A save/load round trip preserves all adapted normalization state exactly.
    copy_path = Path(f"/content/{copy_name}.keras")
    model.save(copy_path, overwrite=True)
    return tf.keras.models.load_model(copy_path, compile=False)

def copy_for_local_training(global_model, client_name):
    local_model = copy_model_exact(global_model, f"local_{client_name}_working_copy")
    for layer in local_model.layers:
        layer.trainable = layer.name == "federated_head"
    compile_model(local_model, learning_rate=LOCAL_LEARNING_RATE)
    return local_model

def validation_gate_snapshot(model):
    probability = predict_probability(model, F_val)
    threshold = select_balanced_threshold(y_val, probability)
    decision = calculate_metrics(y_val, probability, threshold)
    patient_table, mean_patient_roc, mean_patient_pr = mean_patient_ranking(
        y_val, probability, subject_val,
    )
    return {
        "probability": probability,
        "threshold": threshold,
        "pooled_roc_auc": float(decision["roc_auc"]),
        "pooled_pr_auc": float(decision["pr_auc"]),
        "mean_patient_roc_auc": mean_patient_roc,
        "mean_patient_pr_auc": mean_patient_pr,
        "balanced_accuracy": float(decision["balanced_accuracy"]),
        "sensitivity": float(decision["sensitivity"]),
        "specificity": float(decision["specificity"]),
        "predicted_positive_rate": float(decision["predicted_positive_rate"]),
        "probability_std": float(np.std(probability)),
    }

# Keep the original central object as the accepted server model until a proposal passes every gate.
global_model = central_model

# Verify the serialization copy mechanism before any local training or averaging.
copy_probe = copy_model_exact(global_model, "federation_copy_probe")
copy_probe_probability = predict_probability(copy_probe, F_val)
copy_max_probability_error = float(np.max(np.abs(copy_probe_probability - central_validation_probability)))
copy_classification_agreement = float(np.mean(
    (copy_probe_probability >= central_threshold) == (central_validation_probability >= central_threshold)
))
print("Exact-copy max validation probability error:", copy_max_probability_error)
print("Exact-copy validation classification agreement:", copy_classification_agreement)
if copy_max_probability_error > 1e-6 or copy_classification_agreement < 1.0:
    raise RuntimeError("Federation stopped: copied model does not match the central model")
del copy_probe, copy_probe_probability

federation_client_rows = []
federation_acceptance_rows = []
current_gate = validation_gate_snapshot(global_model)

for round_number in range(1, FEDERATED_ROUNDS + 1):
    client_updates = []
    for client_name, features, labels, subjects in [
        ("raspberry_pi", F_pi, y_pi, subject_pi),
        ("renesas", F_ra, y_ra, subject_ra),
    ]:
        local_model = copy_for_local_training(global_model, client_name)
        before = predict_probability(local_model, features)
        local_model.fit(
            features, labels, sample_weight=patient_sample_weights(labels, subjects),
            epochs=LOCAL_EPOCHS, batch_size=128, shuffle=True, verbose=0,
        )
        after = predict_probability(local_model, features)
        head_weights = local_model.get_layer("federated_head").get_weights()
        client_updates.append((len(labels), head_weights))
        federation_client_rows.append({
            "round": round_number, "client": client_name, "windows": len(labels),
            "before_roc_auc": roc_auc_score(labels, before),
            "after_local_roc_auc": roc_auc_score(labels, after),
            "before_pr_auc": average_precision_score(labels, before),
            "after_local_pr_auc": average_precision_score(labels, after),
        })
        del local_model

    total_examples = sum(count for count, _ in client_updates)
    averaged_head = [
        sum(count * weights[index] for count, weights in client_updates) / total_examples
        for index in range(len(client_updates[0][1]))
    ]
    proposed_model = copy_model_exact(global_model, f"federation_round_{round_number}_proposal")
    proposed_model.get_layer("federated_head").set_weights(averaged_head)
    proposed_gate = validation_gate_snapshot(proposed_model)

    gate_checks = {
        "pooled_roc_preserved": proposed_gate["pooled_roc_auc"] >= current_gate["pooled_roc_auc"] - FEDERATION_TOLERANCE,
        "pooled_pr_preserved": proposed_gate["pooled_pr_auc"] >= current_gate["pooled_pr_auc"] - FEDERATION_TOLERANCE,
        "mean_patient_roc_preserved": proposed_gate["mean_patient_roc_auc"] >= current_gate["mean_patient_roc_auc"] - FEDERATION_TOLERANCE,
        "mean_patient_pr_preserved": proposed_gate["mean_patient_pr_auc"] >= current_gate["mean_patient_pr_auc"] - FEDERATION_TOLERANCE,
        "balanced_accuracy_preserved": proposed_gate["balanced_accuracy"] >= current_gate["balanced_accuracy"] - FEDERATION_TOLERANCE,
        "sensitivity_preserved": proposed_gate["sensitivity"] >= current_gate["sensitivity"] - FEDERATION_TOLERANCE,
        "probability_spread_preserved": proposed_gate["probability_std"] >= current_gate["probability_std"] * MIN_FEDERATED_SPREAD_RATIO,
    }
    accepted = bool(all(gate_checks.values()))
    row = {
        "round": round_number, "accepted": accepted,
        **{f"check_{name}": bool(value) for name, value in gate_checks.items()},
    }
    for name in (
        "pooled_roc_auc", "pooled_pr_auc", "mean_patient_roc_auc", "mean_patient_pr_auc",
        "balanced_accuracy", "sensitivity", "specificity", "predicted_positive_rate", "probability_std",
    ):
        row[f"current_{name}"] = current_gate[name]
        row[f"proposed_{name}"] = proposed_gate[name]
    row["tolerance"] = FEDERATION_TOLERANCE
    federation_acceptance_rows.append(row)

    if accepted:
        global_model = proposed_model
        current_gate = proposed_gate
    else:
        del proposed_model
    gc.collect()

federation_client_metrics = pd.DataFrame(federation_client_rows)
federation_acceptance = pd.DataFrame(federation_acceptance_rows)
federation_update_accepted = bool(federation_acceptance.accepted.any())

# Selection uses server validation only. A harmful proposal leaves the central model untouched.
deployment_model = global_model
deployment_source = "accepted_federated_global" if federation_update_accepted else "central_fallback"
deployment_validation_probability = predict_probability(deployment_model, F_val)
deployment_threshold = select_balanced_threshold(y_val, deployment_validation_probability)

deployment_probabilities = {}
deployment_rows = []
for partition, (features, labels, subjects) in evaluation_sets.items():
    probability = predict_probability(deployment_model, features)
    deployment_probabilities[partition] = probability
    deployment_rows.append({"partition": partition, **calculate_metrics(labels, probability, deployment_threshold)})
deployment_evaluation_metrics = pd.DataFrame(deployment_rows).set_index("partition")

deployment_test_probability = deployment_probabilities["final_test"]
federation_test_comparison = pd.DataFrame({
    "central_before_federation": calculate_metrics(y_test, central_test_probability, central_threshold),
    "selected_after_federation_gate": calculate_metrics(y_test, deployment_test_probability, deployment_threshold),
}).T

display(federation_client_metrics.round(4))
print("Federation acceptance diagnostics")
display(federation_acceptance.T)
print("Deployment source:", deployment_source)
print("Validation-calibrated deployment threshold:", round(deployment_threshold, 3))
display(deployment_evaluation_metrics.round(4))
print("Final-test comparison (reporting only; it did not select the model)")
display(federation_test_comparison.round(4))


## 10. Export, quantize, and apply prototype release gates

The validation-selected model is converted to full INT8 using central-training calibration samples. Validation data checks quantization damage. The untouched final test supplies the final report and can mark the artifact as rejected, but it never switches between central and federated models.


In [ ]:
if ARTIFACT_DIR.exists():
    shutil.rmtree(ARTIFACT_DIR)
ARTIFACT_DIR.mkdir(parents=True)

central_model.save(ARTIFACT_DIR / "cinc2018_ecg_spo2_central_model.keras")
deployment_model.save(ARTIFACT_DIR / "cinc2018_ecg_spo2_selected_model.keras")
subject_stats.to_csv(ARTIFACT_DIR / "subject_split_manifest.csv", index=False)
partition_summary.to_csv(ARTIFACT_DIR / "partition_summary.csv")
candidate_results.to_csv(ARTIFACT_DIR / "sensor_ablation_results.csv")
cross_validation_patients.to_csv(ARTIFACT_DIR / "cross_validation_patient_metrics.csv", index=False)
fold_summary.to_csv(ARTIFACT_DIR / "cross_validation_fold_summary.csv", index=False)
fold_ensemble_diagnostics.to_csv(ARTIFACT_DIR / "fold_ensemble_diagnostics.csv")
central_evaluation_metrics.to_csv(ARTIFACT_DIR / "central_evaluation_metrics.csv")
central_patient_metrics.to_csv(ARTIFACT_DIR / "central_patient_metrics.csv", index=False)
federation_client_metrics.to_csv(ARTIFACT_DIR / "federation_client_metrics.csv", index=False)
federation_acceptance.to_csv(ARTIFACT_DIR / "federation_acceptance.csv", index=False)
deployment_evaluation_metrics.to_csv(ARTIFACT_DIR / "selected_model_evaluation_metrics.csv")
federation_test_comparison.to_csv(ARTIFACT_DIR / "federation_final_test_comparison.csv")

head_weights, head_bias = deployment_model.get_layer("federated_head").get_weights()
np.savez_compressed(
    ARTIFACT_DIR / "selected_federated_head.npz",
    weights=head_weights.astype(np.float32), bias=head_bias.astype(np.float32),
    source=np.asarray(deployment_source),
)

# Externalize feature normalization before INT8 conversion. A single INT8 input scale
# cannot accurately represent raw features ranging from fractions to heart-rate values.
normalizer = deployment_model.get_layer("feature_normalization")
deployment_normalization_mean = np.asarray(normalizer.mean).reshape(-1).astype(np.float32)
deployment_normalization_std = np.maximum(
    np.sqrt(np.asarray(normalizer.variance).reshape(-1)),
    tf.keras.backend.epsilon(),
).astype(np.float32)
deployment_normalization_inverse_std = (1.0 / deployment_normalization_std).astype(np.float32)

def normalize_for_deployment(features):
    features = np.asarray(features, dtype=np.float32)
    return ((features - deployment_normalization_mean) * deployment_normalization_inverse_std).astype(np.float32)

F_train_normalized = normalize_for_deployment(F_train)
F_val_normalized = normalize_for_deployment(F_val)
F_test_normalized = normalize_for_deployment(F_test)

normalized_inputs = tf.keras.Input(shape=(len(FEATURE_NAMES),), name="normalized_features")
x = tf.keras.layers.Dense(32, activation="relu", name="shared_dense_1")(normalized_inputs)
x = tf.keras.layers.Dense(16, activation="relu", name="shared_dense_2")(x)
normalized_outputs = tf.keras.layers.Dense(1, activation="sigmoid", name="federated_head")(x)
int8_export_model = tf.keras.Model(normalized_inputs, normalized_outputs, name="normalized_feature_mlp")
for layer_name in ("shared_dense_1", "shared_dense_2", "federated_head"):
    int8_export_model.get_layer(layer_name).set_weights(deployment_model.get_layer(layer_name).get_weights())

normalized_float_validation_probability = predict_probability(int8_export_model, F_val_normalized)
export_equivalence_error = float(np.max(np.abs(
    normalized_float_validation_probability - deployment_validation_probability
)))
print("External-normalization float equivalence error:", export_equivalence_error)
if export_equivalence_error > 1e-5:
    raise RuntimeError("INT8 export stopped: external normalization changed float predictions")

int8_export_model.save(ARTIFACT_DIR / "cinc2018_normalized_input_export_model.keras")
np.savez_compressed(
    ARTIFACT_DIR / "feature_normalization_float32.npz",
    mean=deployment_normalization_mean,
    inverse_std=deployment_normalization_inverse_std,
    feature_names=np.asarray(FEATURE_NAMES),
)

# Use a deterministic class-balanced calibration sample from the full central-training set.
calibration_rng = np.random.default_rng(SEED + 70000)
positive_indices = np.flatnonzero(y_train == 1)
negative_indices = np.flatnonzero(y_train == 0)
per_class = min(750, len(positive_indices), len(negative_indices))
calibration_indices = np.concatenate([
    calibration_rng.choice(positive_indices, size=per_class, replace=False),
    calibration_rng.choice(negative_indices, size=per_class, replace=False),
])
calibration_rng.shuffle(calibration_indices)

def representative_dataset():
    for index in calibration_indices:
        yield [F_train_normalized[index:index + 1]]

converter = tf.lite.TFLiteConverter.from_keras_model(int8_export_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
tflite_model = converter.convert()
tflite_path = ARTIFACT_DIR / "cinc2018_ecg_spo2_selected_int8.tflite"
tflite_path.write_bytes(tflite_model)

vector_count = min(256, len(y_test))
np.savez_compressed(
    ARTIFACT_DIR / "fixed_test_vectors_float32.npz",
    raw_features=F_test[:vector_count],
    normalized_features=F_test_normalized[:vector_count],
    feature_names=np.asarray(FEATURE_NAMES), label=y_test[:vector_count].astype(np.int8),
    subject=subject_test[:vector_count], start_sec=start_test[:vector_count],
    float_probability=deployment_test_probability[:vector_count],
    float_threshold=np.asarray([deployment_threshold], dtype=np.float32),
    source=np.asarray(deployment_source),
)

print("Deployment source:", deployment_source)
print("Parameters:", int8_export_model.count_params())
print("INT8 model size: %.1f KiB" % (len(tflite_model) / 1024))


In [ ]:
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()
input_detail = interpreter.get_input_details()[0]
output_detail = interpreter.get_output_details()[0]

def quantize(values, detail):
    scale, zero = detail["quantization"]
    if scale <= 0:
        raise ValueError("Model input is not fully quantized")
    return np.clip(np.rint(values / scale + zero), -128, 127).astype(np.int8)

input_scale, input_zero = input_detail["quantization"]
output_scale, output_zero = output_detail["quantization"]
if input_scale <= 0 or output_scale <= 0:
    raise ValueError("Model input or output is not fully quantized")

def predict_int8(normalized_features):
    probabilities = []
    for row in range(len(normalized_features)):
        interpreter.set_tensor(input_detail["index"], quantize(normalized_features[row:row + 1], input_detail))
        interpreter.invoke()
        raw = interpreter.get_tensor(output_detail["index"]).reshape(-1)[0]
        probabilities.append((float(raw) - output_zero) * output_scale)
    return np.asarray(probabilities, dtype=np.float32)

int8_validation_probability = predict_int8(F_val_normalized)
int8_test_probability = predict_int8(F_test_normalized)
int8_threshold = select_balanced_threshold(y_val, int8_validation_probability)

float_validation_metrics = calculate_metrics(y_val, deployment_validation_probability, deployment_threshold)
int8_validation_metrics = calculate_metrics(y_val, int8_validation_probability, int8_threshold)
float_final_metrics = calculate_metrics(y_test, deployment_test_probability, deployment_threshold)
int8_final_metrics = calculate_metrics(y_test, int8_test_probability, int8_threshold)

quantization_validation_comparison = pd.DataFrame({
    "float_selected": float_validation_metrics, "int8_selected": int8_validation_metrics,
}).T
quantization_final_comparison = pd.DataFrame({
    "float_selected": float_final_metrics, "int8_selected": int8_final_metrics,
}).T

validation_probability_error = float(np.mean(np.abs(deployment_validation_probability - int8_validation_probability)))
validation_prediction_agreement = float(np.mean(
    (deployment_validation_probability >= deployment_threshold) == (int8_validation_probability >= int8_threshold)
))
final_probability_error = float(np.mean(np.abs(deployment_test_probability - int8_test_probability)))
final_prediction_agreement = float(np.mean(
    (deployment_test_probability >= deployment_threshold) == (int8_test_probability >= int8_threshold)
))

quantization_validation_pass = bool(
    int8_validation_metrics["roc_auc"] >= float_validation_metrics["roc_auc"] - MAX_INT8_VALIDATION_AUC_DROP
    and int8_validation_metrics["pr_auc"] >= float_validation_metrics["pr_auc"] - MAX_INT8_VALIDATION_AUC_DROP
    and int8_validation_metrics["balanced_accuracy"] >= float_validation_metrics["balanced_accuracy"] - MAX_INT8_VALIDATION_DECISION_DROP
    and int8_validation_metrics["sensitivity"] >= float_validation_metrics["sensitivity"] - MAX_INT8_VALIDATION_SENSITIVITY_DROP
    and validation_prediction_agreement >= MIN_INT8_VALIDATION_AGREEMENT
)
final_pr_baseline = float(np.mean(y_test))
final_performance_pass = bool(
    int8_final_metrics["roc_auc"] >= MIN_FINAL_ROC_AUC
    and int8_final_metrics["pr_auc"] >= final_pr_baseline + MIN_FINAL_PR_GAIN
    and int8_final_metrics["balanced_accuracy"] >= MIN_FINAL_BALANCED_ACCURACY
    and int8_final_metrics["sensitivity"] >= MIN_FINAL_SENSITIVITY
)
deployment_approved = bool(quantization_validation_pass and final_performance_pass)
deployment_status = "PROTOTYPE_ACCEPTED" if deployment_approved else "PROTOTYPE_REJECTED"

print("Float threshold:", round(deployment_threshold, 4))
print("Validation-calibrated INT8 threshold:", round(int8_threshold, 4))
print("Validation float vs INT8")
display(quantization_validation_comparison.round(4))
print("Untouched final-test float vs INT8")
display(quantization_final_comparison.round(4))
print("Validation mean probability difference:", round(validation_probability_error, 6))
print("Validation decision agreement after calibration:", round(validation_prediction_agreement, 4))
print("Final decision agreement after calibration:", round(final_prediction_agreement, 4))
print("Quantization validation gate:", "PASS" if quantization_validation_pass else "FAIL")
print("Final performance gate:", "PASS" if final_performance_pass else "FAIL")
print("Artifact status:", deployment_status)

quantization_validation_comparison.to_csv(ARTIFACT_DIR / "float_vs_int8_validation_metrics.csv", index_label="model")
quantization_final_comparison.to_csv(ARTIFACT_DIR / "float_vs_int8_final_test_metrics.csv", index_label="model")
np.savez_compressed(
    ARTIFACT_DIR / "fixed_test_vectors_float32.npz",
    raw_features=F_test[:vector_count], normalized_features=F_test_normalized[:vector_count],
    feature_names=np.asarray(FEATURE_NAMES), label=y_test[:vector_count].astype(np.int8),
    subject=subject_test[:vector_count], start_sec=start_test[:vector_count],
    float_probability=deployment_test_probability[:vector_count],
    int8_probability=int8_test_probability[:vector_count],
    float_threshold=np.asarray([deployment_threshold], dtype=np.float32),
    int8_threshold=np.asarray([int8_threshold], dtype=np.float32),
    input_scale=np.asarray([input_scale], dtype=np.float32), input_zero_point=np.asarray([input_zero], dtype=np.int32),
    source=np.asarray(deployment_source),
)


In [ ]:
def write_c_array(blob, path, variable_name):
    lines = ["#pragma once", "#include <stdint.h>", "", f"static const uint8_t {variable_name}[] = {{"]
    for offset in range(0, len(blob), 12):
        chunk = blob[offset:offset + 12]
        lines.append("    " + ", ".join(f"0x{byte:02x}" for byte in chunk) + ",")
    lines.extend(["};", f"static const unsigned int {variable_name}_len = {len(blob)}u;", ""])
    Path(path).write_text("\n".join(lines), encoding="utf-8")

def c_float_literal(value):
    text = f"{float(value):.9g}"
    if "." not in text and "e" not in text.lower():
        text += ".0"
    return text + "f"

def write_float_array(values, path, variable_name):
    values = np.asarray(values, dtype=np.float32).reshape(-1)
    lines = ["#pragma once", "", f"static const float {variable_name}[{len(values)}] = {{"]
    for offset in range(0, len(values), 6):
        chunk = values[offset:offset + 6]
        lines.append("    " + ", ".join(c_float_literal(value) for value in chunk) + ",")
    lines.extend(["};", ""])
    Path(path).write_text("\n".join(lines), encoding="utf-8")

write_c_array(tflite_model, ARTIFACT_DIR / "cinc2018_ecg_spo2_selected_int8.h", "g_apnea_model")
write_float_array(
    deployment_normalization_mean,
    ARTIFACT_DIR / "feature_normalization_mean.h", "g_feature_normalization_mean",
)
write_float_array(
    deployment_normalization_inverse_std,
    ARTIFACT_DIR / "feature_normalization_inverse_std.h", "g_feature_normalization_inverse_std",
)

safety_checks = pd.DataFrame([
    {"check": "federation_decision_completed", "passed": True,
     "detail": "Update accepted" if federation_update_accepted else "Harmful update rejected; central fallback retained"},
    {"check": "int8_validation_gate", "passed": bool(quantization_validation_pass),
     "detail": f"calibrated agreement={validation_prediction_agreement:.4f}"},
    {"check": "final_roc_auc", "passed": int8_final_metrics["roc_auc"] >= MIN_FINAL_ROC_AUC,
     "detail": f"value={int8_final_metrics['roc_auc']:.4f}, minimum={MIN_FINAL_ROC_AUC:.2f}"},
    {"check": "final_pr_gain", "passed": int8_final_metrics["pr_auc"] >= final_pr_baseline + MIN_FINAL_PR_GAIN,
     "detail": f"value={int8_final_metrics['pr_auc']:.4f}, baseline={final_pr_baseline:.4f}"},
    {"check": "final_balanced_accuracy", "passed": int8_final_metrics["balanced_accuracy"] >= MIN_FINAL_BALANCED_ACCURACY,
     "detail": f"value={int8_final_metrics['balanced_accuracy']:.4f}, minimum={MIN_FINAL_BALANCED_ACCURACY:.2f}"},
    {"check": "final_sensitivity", "passed": int8_final_metrics["sensitivity"] >= MIN_FINAL_SENSITIVITY,
     "detail": f"value={int8_final_metrics['sensitivity']:.4f}, minimum={MIN_FINAL_SENSITIVITY:.2f}"},
])
safety_checks.to_csv(ARTIFACT_DIR / "prototype_release_gates.csv", index=False)
display(safety_checks)

metadata = {
    "dataset": "PhysioNet/CinC Challenge 2018 v1.0.0", "dataset_subject_limit": SUBJECT_LIMIT,
    "successful_subjects": len(subject_stats), "cache_version": CACHE_VERSION,
    "input_sensors": ["ECG", "SaO2"], "feature_names": FEATURE_NAMES,
    "model_input": "21 externally standardized ECG and SaO2 features",
    "feature_normalization": "(raw_feature - mean) * inverse_std",
    "window_seconds": WINDOW_SEC, "stride_seconds": STRIDE_SEC,
    "label_rule": f"apnea or hypopnea overlap >= {MIN_EVENT_OVERLAP_SEC} seconds",
    "central_training": "central_train with server-validation early stopping",
    "central_best_epoch": central_best_epoch, "deployment_source": deployment_source,
    "federation_update_accepted": federation_update_accepted,
    "federation_acceptance_source": "server validation ranking, decisions, and probability spread",
    "float_threshold": deployment_threshold, "int8_threshold": int8_threshold,
    "threshold_source": "separate server-validation calibration for float and INT8",
    "final_test_used_for_model_selection": False,
    "model_parameters": int(int8_export_model.count_params()),
    "federated_head_parameters": int(deployment_model.get_layer("federated_head").count_params()),
    "federated_rounds_attempted": FEDERATED_ROUNDS, "local_epochs": LOCAL_EPOCHS,
    "local_learning_rate": LOCAL_LEARNING_RATE, "tflite_size_bytes": len(tflite_model),
    "tflite_input_scale": float(input_scale), "tflite_input_zero_point": int(input_zero),
    "tflite_output_scale": float(output_scale), "tflite_output_zero_point": int(output_zero),
    "external_normalization_equivalence_error": export_equivalence_error,
    "validation_float_int8_probability_error": validation_probability_error,
    "validation_float_int8_calibrated_agreement": validation_prediction_agreement,
    "final_float_int8_probability_error": final_probability_error,
    "final_float_int8_calibrated_agreement": final_prediction_agreement,
    "quantization_validation_pass": quantization_validation_pass,
    "final_performance_pass": final_performance_pass,
    "deployment_status": deployment_status, "seed": SEED,
}
(ARTIFACT_DIR / "deployment_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

drive_artifact_dir = DRIVE_PROJECT_DIR / "latest_artifacts"
artifacts_synced = False
if drive_available():
    try:
        if drive_artifact_dir.exists():
            shutil.rmtree(drive_artifact_dir)
        shutil.copytree(ARTIFACT_DIR, drive_artifact_dir)
        artifacts_synced = True
    except OSError as error:
        print("Drive artifact copy postponed:", error)
archive_path = shutil.make_archive("/content/cinc2018_apnea_artifacts", "zip", ARTIFACT_DIR)
if artifacts_synced:
    print("Saved persistent artifacts to:", drive_artifact_dir)
else:
    print("WARNING: artifacts are local only; download the ZIP before the runtime ends.")
print("Created:", archive_path)
print("FINAL STATUS:", deployment_status)

from google.colab import files
files.download(archive_path)


## Final decision rule

The final test never chooses between the central and federated models. Server validation controls early stopping, federation acceptance, threshold calibration, and the INT8 damage check. Final-test metrics only determine whether the already-selected prototype satisfies the declared release gates.

Use identical fixed vectors on the Pi and Renesas for latency, jitter, RAM/flash, energy, and numerical-agreement comparisons. An artifact marked \`PROTOTYPE_REJECTED\` is retained for diagnosis but must not support an accuracy claim or final board demonstration.
